# Day 36 - Interval DP and tree DP: choosing the split point instead of the next item

Day 33 filled a one-dimensional table, day 34 made it two-dimensional, and day 35
used that square table for LIS and LCS.  In all of those, the table was indexed by
a *prefix*: "the best answer using the first i items".

Today the index changes shape twice.  Interval DP indexes a table by a *segment*
`[i, j]` and the choice at each cell is where to split it.  Tree DP indexes by a
*subtree* and the choice is what each child reports upward.  Both are still just
"solve smaller things first", but "smaller" stops meaning "shorter prefix", and
that is where the two classic silent bugs of the day come from:

* filling an interval table in the natural `i`-then-`j` order reads cells that
  have not been computed yet - and returns a smaller number without raising,
* giving each tree node one number instead of a pair loses the one fact the
  parent needed.

In [1]:
from functools import lru_cache
import sys

## 1. Matrix chain multiplication

Multiplying an `r x c` matrix by a `c x s` matrix costs `r * c * s` scalar
multiplications.  Matrix multiplication is associative, so the *result* does not
depend on how you bracket the chain - but the *cost* very much does.

Here is a direct cost function for one explicit parenthesisation, so the DP has
something honest to be checked against.

In [2]:
def chain_cost(dims, order):
    """Cost of one explicit parenthesisation, for checking the DP by hand.

    `order` is a nested tuple of matrix indices, e.g. ((0, 1), (2, 3)).
    Returns (rows, cols, scalar multiplications).
    """
    if isinstance(order, int):
        return dims[order], dims[order + 1], 0
    r1, c1, k1 = chain_cost(dims, order[0])
    r2, c2, k2 = chain_cost(dims, order[1])
    assert c1 == r2, 'shapes do not line up'
    return r1, c2, k1 + k2 + r1 * c1 * c2

DIMS = [40, 20, 30, 10, 30]     # A:40x20  B:20x30  C:30x10  D:10x30
print('((AB)(CD)) ->', chain_cost(DIMS, ((0, 1), (2, 3)))[2], 'multiplications')
print('(A(B(CD))) ->', chain_cost(DIMS, (0, (1, (2, 3))))[2], 'multiplications')
print('((A(BC))D) ->', chain_cost(DIMS, ((0, (1, 2)), 3))[2], 'multiplications')

((AB)(CD)) -> 69000 multiplications
(A(B(CD))) -> 51000 multiplications
((A(BC))D) -> 26000 multiplications


The three bracketings above compute exactly the same 40x30 matrix, and the worst
one costs more than twice the best one.  With `n` matrices the number of
bracketings is the Catalan number `C(n-1)`, so enumerating them is hopeless past
about fifteen matrices.

The DP: a product of matrices `i..j` is, at the very top, a product of two
things - `i..k` and `k+1..j`.  Split on that last multiplication and both pieces
are strictly shorter intervals.

In [3]:
def matrix_chain(dims):
    """dp[i][j] = cheapest way to multiply matrices i..j (inclusive).

    A product of matrices i..j is, at the very top, a product of exactly two
    things: i..k and k+1..j.  So the recurrence splits on that last
    multiplication:

        dp[i][j] = min over k of  dp[i][k] + dp[k+1][j] + rows(i)*cols(k)*cols(j)

    Both sub-intervals are strictly shorter than i..j, which is the whole
    reason the loop is over *length*.
    """
    n = len(dims) - 1
    dp = [[0] * n for _ in range(n)]
    split = [[-1] * n for _ in range(n)]
    for length in range(2, n + 1):                 # <- the outer loop is length
        for i in range(n - length + 1):
            j = i + length - 1
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                if cost < dp[i][j]:
                    dp[i][j] = cost
                    split[i][j] = k
    return dp, split

def parens(split, i, j, names=None):
    """Rebuild the parenthesisation from the split points."""
    if names is None:
        names = [chr(ord('A') + t) for t in range(len(split))]
    if i == j:
        return names[i]
    k = split[i][j]
    return '(' + parens(split, i, k, names) + parens(split, k + 1, j, names) + ')'

def chain_order(split, i, j):
    """The same reconstruction, as a nested tuple for chain_cost()."""
    if i == j:
        return i
    k = split[i][j]
    return (chain_order(split, i, k), chain_order(split, k + 1, j))

dp, split = matrix_chain(DIMS)
best = dp[0][len(DIMS) - 2]
print('cheapest      =', f'{best:,}')
print('bracketing    =', parens(split, 0, len(DIMS) - 2))
print('re-checked    =', f'{chain_cost(DIMS, chain_order(split, 0, len(DIMS) - 2))[2]:,}')

cheapest      = 26,000
bracketing    = ((A(BC))D)
re-checked    = 26,000


### The loop order is the whole algorithm

`dp[i][j]` reads `dp[i][k]` (same row, to the left) and `dp[k+1][j]` (a *lower*
row, same column).  The natural double loop `for i ... for j ...` walks the rows
top to bottom, so when it computes `dp[0][3]` the cell `dp[k+1][3]` below it is
still zero.  Nothing raises - zero is a perfectly good number - and the answer
comes out too small.

The fix is to loop over interval **length** first.  Then every sub-interval is
already filled, because it is shorter.

In [4]:
def matrix_chain_naive_order(dims):
    """The same body with the natural i-then-j loops.  Wrong, and silent.

    dp[i][j] reads dp[k+1][j] with k+1 > i - a *later* row, which has not been
    filled yet and is still 0.  Nothing raises; the answer is simply too small.
    """
    n = len(dims) - 1
    dp = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1, n):
            dp[i][j] = float('inf')
            for k in range(i, j):
                cost = dp[i][k] + dp[k + 1][j] + dims[i] * dims[k + 1] * dims[j + 1]
                dp[i][j] = min(dp[i][j], cost)
    return dp[0][n - 1]

def matrix_chain_fill_order(n):
    """The (i, j) cells in the order the correct loop visits them."""
    out = []
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            out.append((i, i + length - 1))
    return out

print('correct (length-first) =', f'{best:,}')
print('naive   (i-then-j)     =', f'{matrix_chain_naive_order(DIMS):,}')
print()
print('fill order:', matrix_chain_fill_order(len(DIMS) - 1))

correct (length-first) = 26,000
naive   (i-then-j)     = 20,000

fill order: [(0, 1), (1, 2), (2, 3), (0, 2), (1, 3), (0, 3)]


Read that fill order as a triangle: the whole second diagonal first, then the
third, and only at the very end the top-right corner `(0, 3)` that holds the
answer.  Every interval DP in this notebook has that same shape.

## 2. LeetCode 312 - Burst Balloons

**The task.** A row of balloons each carries a number.  You pop them one at a
time; popping a balloon earns you the product of its own number and the numbers
of the balloons immediately to its left and right *at that moment*.  Balloons
that have already been popped are gone, so neighbours change as you go.  When a
popped balloon has no neighbour on a side, treat that missing neighbour as a 1.
Maximise the total coins collected after every balloon has been popped.

**Input / output.** A list of integers `nums`; return one integer, the maximum
total.

**Example.** `nums = [3, 1, 5, 8]`.  Popping in the order 1, 5, 3, 8 earns
`3*1*5 = 15`, then `3*5*8 = 120`, then `1*3*8 = 24`, then `1*8*1 = 8`, for a
total of 167, which is the best possible.

**Constraints.** `1 <= n <= 300` and `0 <= nums[i] <= 100`.  An `O(n^3)` interval
DP is about 27 million operations at the top end - comfortable - while the
`n!` orderings are not, so the bound is effectively telling you the intended
solution is a cubic DP over intervals.

[leetcode.com/problems/burst-balloons](https://leetcode.com/problems/burst-balloons/)

The instinctive move is to choose which balloon to pop **first**.  It does not
decompose: after `k` is gone, the left half's right-hand neighbour is a balloon
from the right half, so the two sides are not independent and the coins the
recurrence assumes are not the coins you actually get.

In [5]:
def burst_first_wrong(nums):
    """The instinctive recurrence: choose which balloon to pop FIRST.

    Pop k first, collect a[k-1] * a[k] * a[k+1] using the neighbours it has
    *right now*, then solve the two sides separately.  It does not decompose:
    once k is gone the left side's right-hand neighbour is a balloon from the
    right side, so the two halves are not independent and the coins they can
    still earn are not the ones this recurrence assumes.  It returns 98 on the
    example instead of 167 - no exception, just a smaller number.
    """
    a = [1] + list(nums) + [1]

    @lru_cache(maxsize=None)
    def go(i, j):                          # inclusive, in the padded array
        if i > j:
            return 0
        return max(a[k - 1] * a[k] * a[k + 1] + go(i, k - 1) + go(k + 1, j)
                   for k in range(i, j + 1))

    return go(1, len(nums))

BALLOONS = [3, 1, 5, 8]
print('split-on-first  ->', burst_first_wrong(BALLOONS))

split-on-first  -> 98


Choose the balloon popped **last** instead.  If `k` is last inside the open
interval `(i, j)`, then by the time `k` pops, everything strictly between `i`
and `j` is already gone - so its neighbours are exactly the fixed walls `i` and
`j`, which do not depend on anything that happened inside.  Now the two sides
really are independent.

In [6]:
def burst_last(nums):
    """The correct recurrence: choose which balloon is popped LAST in (i, j).

    If k is last inside the open interval (i, j), then by the time k pops
    everything strictly between i and j is gone, so its neighbours are exactly
    the fixed walls i and j.  The two sides never interact, and the
    subproblems are genuinely independent.
    """
    a = [1] + list(nums) + [1]
    n = len(a)
    dp = [[0] * n for _ in range(n)]
    choice = [[-1] * n for _ in range(n)]
    for length in range(2, n):                 # length = j - i, at least 2
        for i in range(n - length):
            j = i + length
            for k in range(i + 1, j):
                cand = dp[i][k] + dp[k][j] + a[i] * a[k] * a[j]
                if cand > dp[i][j]:
                    dp[i][j] = cand
                    choice[i][j] = k
    return dp[0][n - 1], dp, choice

def burst_order(choice, i, j):
    """The actual popping order, read out of the choice table."""
    k = choice[i][j]
    if k == -1:
        return []
    return burst_order(choice, i, k) + burst_order(choice, k, j) + [k]

def burst_brute(nums):
    """Every popping order, for small inputs.  The ground truth."""
    best = 0
    def go(cur, gained):
        nonlocal best
        if not cur:
            best = max(best, gained)
            return
        for i in range(len(cur)):
            left = cur[i - 1] if i else 1
            right = cur[i + 1] if i + 1 < len(cur) else 1
            go(cur[:i] + cur[i + 1:], gained + left * cur[i] * right)
    go(list(nums), 0)
    return best

best, dp, choice = burst_last(BALLOONS)
order = burst_order(choice, 0, len(BALLOONS) + 1)
padded = [1] + BALLOONS + [1]
print('split-on-last   ->', best)
print('popping order   ->', [padded[k] for k in order])
print('brute force     ->', burst_brute(BALLOONS))

split-on-last   -> 167
popping order   -> [1, 5, 3, 8]
brute force     -> 167


## 3. LeetCode 1547 - Minimum Cost to Cut a Stick

**The task.** A wooden stick of length `n` is marked at several positions.  You
must make a cut at every marked position, in any order you like.  Cutting a
piece costs the length of *that piece*, so the order changes the bill.  Find the
cheapest total.

**Input / output.** An integer `n` and a list `cuts` of positions strictly
between 0 and `n`; return the minimum total cost.

**Example.** `n = 7`, `cuts = [1, 3, 4, 5]`.  Cutting in the given order costs
20; a better order costs 16.

**Constraints.** `2 <= n <= 10^6` but `cuts.length <= 100`.  Note which of the
two is small: the table is indexed by *cut positions*, not by stick length, so
the DP is `O(m^3)` in the number of cuts and the huge `n` is irrelevant.

[leetcode.com/problems/minimum-cost-to-cut-a-stick](https://leetcode.com/problems/minimum-cost-to-cut-a-stick/)

This one is the mirror image of balloons: here the **first** cut inside a piece
is the separable one, because it splits the piece into two that never interact
again.  Same table, same length-first loop; only the direction of "which choice
decomposes" flips.

In [7]:
def cut_cost_in_order(n, cuts):
    """Cost of performing the cuts in the order they are given - the baseline."""
    pieces = [(0, n)]
    total = 0
    for c in cuts:
        for idx, (lo, hi) in enumerate(pieces):
            if lo < c < hi:
                total += hi - lo
                pieces[idx:idx + 1] = [(lo, c), (c, hi)]
                break
    return total

def min_cost_cuts(n, cuts):
    """Add the two ends as fake cuts, sort, and it is matrix chain again.

    dp[i][j] = cheapest way to make every cut strictly between points i and j.
    The *first* cut made inside a piece costs the whole length of that piece,
    and splits it into two pieces that never interact again - so here the
    natural "which one first" framing is the one that decomposes.
    """
    pts = sorted(cuts) + [0, n]
    pts.sort()
    m = len(pts)
    dp = [[0] * m for _ in range(m)]
    for length in range(2, m):
        for i in range(m - length):
            j = i + length
            dp[i][j] = min(dp[i][k] + dp[k][j] for k in range(i + 1, j)) + pts[j] - pts[i]
    return dp[0][m - 1]

STICK_N, CUTS = 7, [1, 3, 4, 5]
print('cutting in the given order =', cut_cost_in_order(STICK_N, CUTS))
print('cheapest order             =', min_cost_cuts(STICK_N, CUTS))

cutting in the given order = 20
cheapest order             = 16


## 4. LeetCode 516 - Longest Palindromic Subsequence

**The task.** Given a string, find the length of the longest subsequence of it
that reads the same forwards and backwards.  A subsequence keeps the original
order but may drop any characters.

**Input / output.** A string `s`; return one integer.

**Example.** `s = "character"`.  The longest palindromic subsequence is
`"carac"`, so the answer is 5.

**Constraints.** `1 <= s.length <= 1000`, lowercase letters.  A thousand
characters means an `O(n^2)` table of a million cells is fine, and anything
exponential over subsequences is not.

[leetcode.com/problems/longest-palindromic-subsequence](https://leetcode.com/problems/longest-palindromic-subsequence/)

Two ways in, and it is worth seeing both.  Directly, as interval DP: if the ends
match they both join the palindrome, otherwise drop one end and take the better
side.  Or, using day 35: a palindromic subsequence of `s` is a subsequence that
also appears in `reversed(s)`, so the answer is `LCS(s, reversed(s))`.

In [8]:
def lps_interval(s):
    """Interval DP: dp[i][j] = the LPS of s[i..j]."""
    n = len(s)
    if n == 0:
        return 0, ''
    dp = [[0] * n for _ in range(n)]
    for i in range(n):
        dp[i][i] = 1
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length - 1
            if s[i] == s[j]:
                dp[i][j] = dp[i + 1][j - 1] + 2
            else:
                dp[i][j] = max(dp[i + 1][j], dp[i][j - 1])
    # backtrack for the palindrome itself
    out, i, j = [], 0, n - 1
    while i < j:
        if s[i] == s[j]:
            out.append(s[i]); i += 1; j -= 1
        elif dp[i + 1][j] >= dp[i][j - 1]:
            i += 1
        else:
            j -= 1
    mid = [s[i]] if i == j else []
    return dp[0][n - 1], ''.join(out) + ''.join(mid) + ''.join(reversed(out))

def lps_via_lcs(s):
    """Day 35's LCS, run against the reversed string.  Same number.

    A palindromic subsequence of s is a subsequence that also appears in
    reversed(s), so LPS(s) = LCS(s, reversed(s)).  Identical O(n^2), and it is
    worth knowing because it turns a "new" problem into yesterday's.
    """
    x, y = s, s[::-1]
    dp = [[0] * (len(y) + 1) for _ in range(len(x) + 1)]
    for i in range(1, len(x) + 1):
        for j in range(1, len(y) + 1):
            if x[i - 1] == y[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[len(x)][len(y)]

S = 'character'
n, pal = lps_interval(S)
print('interval DP ->', n, repr(pal))
print('via LCS     ->', lps_via_lcs(S))

interval DP -> 5 'carac'
via LCS     -> 5


## 5. Tree DP - LeetCode 337, House Robber III

**The task.** A thief robs houses arranged as a binary tree.  Each house holds
some money, but the alarm goes off if two *directly connected* houses are both
robbed - that is, a node and either of its children.  Maximise the money taken.

**Input / output.** The root of a binary tree of non-negative integers; return
one integer.

**Example.** Root 3, with left child 2 whose right child is 3, and right child 3
whose right child is 1.  Robbing the root plus the two grandchildren gives
`3 + 3 + 1 = 7`, which is optimal.

**Constraints.** Up to 10^4 nodes, values in `[0, 10^4]`.  Small enough that
recursion survives on LeetCode's trees, large enough that trying every subset is
out of the question.

[leetcode.com/problems/house-robber-iii](https://leetcode.com/problems/house-robber-iii/)

The important part is the shape of the state.  "The best for this subtree" is one
number, and it is not enough: the parent's legal moves depend on whether *this*
node was used, and that single number has already thrown that away.  Return a
pair instead - the best with this node taken, and the best with it skipped.

In [9]:
class Node:
    __slots__ = ('val', 'left', 'right')

    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def build_tree():
    """     3
          /   \\
         2     3
          \\     \\
           3     1        LeetCode 337's first example: the answer is 7.
    """
    return Node(3, Node(2, None, Node(3)), Node(3, None, Node(1)))

def rob_tree(root):
    """LeetCode 337.  Each node returns a PAIR: (robbed, skipped).

    One number per node is not enough state - "the best for this subtree"
    cannot tell the parent whether this node was used, and the parent's
    decision depends on exactly that.  Two numbers make the states independent
    again.  Same lesson as day 35's "dp[i] must end at i".
    """
    def go(node):
        if node is None:
            return 0, 0
        lr, ls = go(node.left)
        rr, rs = go(node.right)
        robbed = node.val + ls + rs           # take me: children must be skipped
        skipped = max(lr, ls) + max(rr, rs)   # skip me: children are free
        return robbed, skipped
    return max(go(root))

def rob_pair(node):
    """The (robbed, skipped) pair for one node - what `rob_tree` computes
    internally.  Exposed so the figures can print real numbers."""
    if node is None:
        return 0, 0
    lr, ls = rob_pair(node.left)
    rr, rs = rob_pair(node.right)
    return node.val + ls + rs, max(lr, ls) + max(rr, rs)

def rob_tree_one_state(root):
    """The wrong version: one number per node, "the best for this subtree".

    That single number cannot say whether the subtree's own root was used, so
    the parent has no way to know that adding itself is illegal - and with
    positive values `max` always prefers taking the node, so every node ends up
    taken, parents and children together.  It returns 12 instead of 7, which is
    just "the sum of everything" wearing a DP costume.
    """
    def go(node):
        if node is None:
            return 0
        below = go(node.left) + go(node.right)
        return max(node.val + below, below)
    return go(root)

root = build_tree()
print('pair of states (take, skip) ->', rob_tree(root))
print('one number per node         ->', rob_tree_one_state(root))
for name, nd in [('root', root), ('left', root.left), ('right', root.right)]:
    print(f'  {name:5s} val={nd.val}  (take, skip) = {rob_pair(nd)}')

pair of states (take, skip) -> 7
one number per node         -> 12
  root  val=3  (take, skip) = (7, 6)
  left  val=2  (take, skip) = (2, 3)
  right val=3  (take, skip) = (3, 1)


The single-state version returns 12, which is just the sum of every node - with
positive values, `max(val + below, below)` always prefers taking the node, so
every node gets taken, parents and children together.  It is a wrong answer
dressed as a DP, and it never raises.

The same "what a node returns is not what it scores" split drives two more
classics.

In [10]:
def max_path_sum(root):
    """LeetCode 124.  The same split: what a node RETURNS is not what it SCORES.

    A node returns the best downward path through it (at most one child), but
    the answer it contributes may bend at the node and use both children.
    Mixing those two up is the entire difficulty of the problem.
    """
    best = float('-inf')

    def down(node):
        nonlocal best
        if node is None:
            return 0
        left = max(down(node.left), 0)         # a negative branch is worth skipping
        right = max(down(node.right), 0)
        best = max(best, node.val + left + right)   # bend here
        return node.val + max(left, right)          # go through here
    down(root)
    return best

def diameter(root):
    """The same pattern again: return depth, score depth(left)+depth(right)."""
    best = 0

    def depth(node):
        nonlocal best
        if node is None:
            return 0
        l, r = depth(node.left), depth(node.right)
        best = max(best, l + r)
        return 1 + max(l, r)
    depth(root)
    return best

print('LeetCode 124 max path sum =', max_path_sum(root))
print('tree diameter (in edges)  =', diameter(root))

LeetCode 124 max path sum = 12
tree diameter (in edges)  = 4


In `max_path_sum`, a node *returns* the best downward path through it, which may
use at most one child, because the parent will extend it.  But the answer a node
*contributes* may bend at that node and use both children.  Two different
quantities, one function - and confusing them is the entire difficulty of the
problem.  `diameter` is the identical pattern with depth instead of sum.

## 6. Tree knapsack, and why its double loop is not cubic

Now combine the two halves of the day: a knapsack (day 34) whose items form a
tree.  A course may only be taken together with its prerequisite, so choosing a
node commits you to its ancestors.  Merge each child's table into the parent's.

In [11]:
def tree_knapsack(children, value, root, budget):
    """Pick `budget` courses; a course may only be taken with its prerequisite.

    dp[v] is a list: dp[v][t] = best value from v's subtree taking exactly t
    courses, v included.  Merging a child is a small knapsack over the sizes
    already accumulated - and the loop bound `min(size[v], budget)` is not
    an optimisation, it is what makes the whole thing O(n * budget) instead of
    O(n * budget^2).
    """
    size = {}

    def go(v):
        dp = [0] * (budget + 1)
        for t in range(1, budget + 1):
            dp[t] = value[v]
        size[v] = 1
        for c in children[v]:
            cdp = go(c)
            nd = list(dp)
            for t in range(min(size[v], budget), 0, -1):
                for u in range(1, min(size[c], budget - t) + 1):
                    if dp[t] + cdp[u] > nd[t + u]:
                        nd[t + u] = dp[t] + cdp[u]
            dp = nd
            size[v] += size[c]
        return dp

    return go(root)[budget], size

COURSES = {0: [1, 2], 1: [3, 4], 2: [5], 3: [], 4: [], 5: []}
CREDITS = [0, 2, 3, 4, 1, 5]      # course 0 is a virtual free root
for m in range(1, 6):
    best, _ = tree_knapsack(COURSES, CREDITS, 0, m + 1)
    print(f'take {m} real course(s) -> {best} credits')

take 1 real course(s) -> 3 credits
take 2 real course(s) -> 8 credits
take 3 real course(s) -> 10 credits
take 4 real course(s) -> 14 credits
take 5 real course(s) -> 15 credits


The merge looks like a triple loop and therefore like `O(n * budget^2)` at best.
It is not, and the reason is a counting argument worth remembering: the inner
work at a node is `size(child) * size(merged so far)`, which is exactly the
number of pairs `(u, v)` whose lowest common ancestor is that node.  Summed over
the whole tree, every pair of nodes is charged exactly once - so the total is
`C(n, 2)`, and the merge is `O(n^2)` overall (and `O(n * budget)` once the loop
bounds are capped by the budget).

In [12]:
def pair_count(children, root):
    """Why the naive-looking merge is O(n^2): every PAIR of nodes is counted once.

    The inner double loop at a node costs size(child) * size(already merged) -
    which is exactly the number of (u, v) pairs whose lowest common ancestor is
    this node.  Summed over the tree, every pair is charged exactly once, so
    the total is C(n, 2).
    """
    total = 0
    size = {}

    def go(v):
        acc = 1
        for c in children[v]:
            sc = go(c)
            nonlocal total
            total += acc * sc          # pairs whose LCA is v
            acc += sc
        size[v] = acc
        return acc

    n = go(root)
    return total, n * (n - 1) // 2

total, expected = pair_count(COURSES, 0)
print('inner-loop units charged =', total)
print('C(n, 2)                  =', expected)

inner-loop units charged = 15
C(n, 2)                  = 15


## 7. The depth trap

Tree DP is naturally recursive, and Python's default recursion limit is 1000.
A tree that happens to be a path - a linked list wearing a tree costume - blows
straight through it.  This is the one *loud* failure of the day, and the fix is
a post-order walk without the call stack: push nodes, record the order, then
process it in reverse so every child is finished before its parent.

In [13]:
def path_tree(n):
    """children lists for 0 -> 1 -> 2 -> ... -> n-1, the worst case for depth."""
    return {i: ([i + 1] if i + 1 < n else []) for i in range(n)}

def subtree_sums_recursive(children, value, root):
    def go(v):
        return value[v] + sum(go(c) for c in children[v])
    return go(root)

def subtree_sums_iterative(children, value, root):
    """Post-order without the call stack: push, then pop in reverse.

    The first pass produces a valid topological order of the tree (parents
    before children); reversing it gives children before parents, which is
    exactly what a tree DP needs.  No recursion limit, no C stack.
    """
    order, stack = [], [root]
    while stack:
        v = stack.pop()
        order.append(v)
        stack.extend(children[v])
    total = {}
    for v in reversed(order):
        total[v] = value[v] + sum(total[c] for c in children[v])
    return total[root]

N = 60000
kids = path_tree(N)
vals = [1] * N
try:
    print(subtree_sums_recursive(kids, vals, 0))
except RecursionError as exc:
    print('recursive  -> RecursionError:', exc)
print('iterative  ->', subtree_sums_iterative(kids, vals, 0))

recursive  -> RecursionError: maximum recursion depth exceeded
iterative  -> 60000


## 8. Complexity

| Problem | Time | Space |
| --- | --- | --- |
| matrix chain | `O(n^3)` | `O(n^2)` |
| burst balloons / cut a stick | `O(n^3)` | `O(n^2)` |
| longest palindromic subsequence | `O(n^2)` | `O(n^2)` |
| house robber III / max path sum / diameter | `O(n)` | `O(h)` |
| tree knapsack | `O(n * budget)` | `O(n * budget)` |

Interval DP has `O(n^2)` cells and each one scans up to `n` split points, hence
the cube.  Tree DP visits every node once; only the knapsack pays extra, and its
extra is the pair-counting argument above.

## 9. Checks

In [14]:
dp, split = matrix_chain(DIMS)
assert dp[0][3] == 26000 and parens(split, 0, 3) == '((A(BC))D)'
assert chain_cost(DIMS, chain_order(split, 0, 3))[2] == 26000
assert matrix_chain_naive_order(DIMS) == 20000            # silently too small
assert matrix_chain_fill_order(4)[0] == (0, 1) and matrix_chain_fill_order(4)[-1] == (0, 3)

assert burst_last(BALLOONS)[0] == 167 == burst_brute(BALLOONS)
assert burst_first_wrong(BALLOONS) == 98
assert cut_cost_in_order(STICK_N, CUTS) == 20 and min_cost_cuts(STICK_N, CUTS) == 16

assert lps_interval('character') == (5, 'carac')
assert lps_via_lcs('character') == 5
assert lps_interval('bbbab')[0] == 4 and lps_interval('cbbd')[0] == 2

assert rob_tree(root) == 7 and rob_tree_one_state(root) == 12
assert rob_pair(root) == (7, 6)
assert max_path_sum(root) == 12 and diameter(root) == 4

assert tree_knapsack(COURSES, CREDITS, 0, 3)[0] == 8
assert tree_knapsack(COURSES, CREDITS, 0, 5)[0] == 14
assert pair_count(COURSES, 0) == (15, 15)

assert subtree_sums_iterative(path_tree(60000), [1] * 60000, 0) == 60000
print('all assertions passed')

all assertions passed
